# Architecting Autonomous Intelligence
### Two hours. Seven steps. One agent that starts out very sure of itself.

Your college admin office gets tickets like *"I can't download my hall ticket,
my exam is on Monday."* Somebody has to read each one and decide who deals
with it. Today you build the thing that does the deciding.

It will not go smoothly, and that is the plan. By step 2 your agent will
produce a perfectly formatted, fully validated, entirely **invented** answer
about a student it has never looked up. Fixing that is most of the workshop.

Nothing to install. No Python, no `make`, no "works on my machine". It all
runs here.

**You need exactly one thing:** a Gemini API key. Free, no card, two minutes —
<https://aistudio.google.com/apikey>

> ### Get your own key. Seriously.
> Rate limits count per Google Cloud project, not per key. Thirty people on
> one key get about **sixteen requests each** — and a single ticket costs the
> agent 5–15 model calls. Do the arithmetic: a shared key dies before the
> first coffee break, and it takes the whole room with it.

---

## How this works

Each step is a git branch, and each one adds exactly one idea.

The good bit is not any single branch — it is the **diff between two of
them**. One concept, nothing else mixed in, no hunting through a file for the
line that matters. Every step below shows you that diff before it runs
anything. Read them. That is where the workshop actually is.

The whole repo is at <https://github.com/soupforcode/agent-demo> if you would rather
work locally; see `docs/00-setup.md`.

---

## Setup — run this once

Clones the repo, installs everything, and builds a small fake college database
— 12 students with their fees, hostel rooms and exam records. All invented,
none of it anybody's real data.

A minute or two. Good time to go and get your API key if you haven't.

In [ ]:
# Safe to re-run. "Runtime -> Restart session" clears the Python kernel but
# NOT /content, so the clone survives a restart — and an unguarded clone would
# greet you with a red "fatal: destination path already exists" for no reason.
![ -d /content/agent-demo ] || git clone -q https://github.com/soupforcode/agent-demo.git /content/agent-demo
%cd /content/agent-demo
%pip install -q -e ".[dev]"
!python -m college_agent.data.seed

> **If the next cell fails with an import error**, Colab had an older version
> of something already loaded. Go to **Runtime → Restart session**, then run
> the cells again *starting from the API key cell below* — the clone and
> install survive a restart, so you do not need to repeat them.

### Your API key

Use the **secrets panel**, not a literal in a cell. Anything typed into a cell
is saved inside the notebook and goes wherever the notebook goes — which is a
genuinely common way to publish a working key to the internet by accident.

1. Click the **🔑 key icon** in the left sidebar.
2. **+ Add new secret**, name it exactly `GOOGLE_API_KEY`.
3. Paste your key as the value, and turn on **Notebook access**.

> **It is `GOOGLE_API_KEY`, not `GEMINI_API_KEY`.** Google's own quickstart
> tells you the second one. The framework only ever reads the first. This one
> mismatch has eaten more workshop time than every other setup problem
> combined, and the error it throws points nowhere near the cause.

In [ ]:
import os

try:
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    # Not on Colab, or the secret is not set — ask for it without echoing.
    import getpass

    os.environ["GOOGLE_API_KEY"] = getpass.getpass("GOOGLE_API_KEY: ")

# The variable is GOOGLE_API_KEY, not GEMINI_API_KEY. Google's own quickstart
# says the latter; the framework only reads the former. This costs people
# twenty minutes roughly every time.
print("key set:", bool(os.environ.get("GOOGLE_API_KEY", "").strip()))

### Check it works

One real call to the model. If this passes, everything below will run — and if it doesn't, it names the part that is unhappy instead of making you guess.

In [ ]:
!python scripts/preflight.py

---

# Step 1 of 7 — An agent

*Confidence: total. Evidence: none. We start here so the rest lands.*

**New here:** A model, some instructions, and nothing else. No tools, no schema.

- It answers in prose — readable, but no system can route on it.
- It has no way to look up CS22B007, so anything it says about that
- student is invented. Both problems get fixed, one per step.

In [ ]:
!git switch -q step-1-agent
!git --no-pager log --oneline -1

In [ ]:
!python -m college_agent.agent

---

# Step 2 of 7 — A contract

*Same wrong answer, now beautifully typed.*

**New here:** output_schema=TriageResult — one parameter.

- You now get a validated object you could route, count and test.
- It is also still invented — the agent has no tools yet. Structure
- buys parseability, not truth, and it makes a wrong answer look
- considerably more authoritative than prose did.

**The diff is the lesson.** Everything that changed between step 1 and step 2. The file summary comes first — read that, pick the file that looks interesting, then find it in the diff underneath.

In [ ]:
!git switch -q step-2-structured
!git --no-pager diff --stat step-1-agent step-2-structured
print()
!git --no-pager diff step-1-agent step-2-structured -- src labs evals tests ':!src/college_agent/data' ':!src/college_agent/kb'

In [ ]:
!python -m college_agent.agent

---

# Step 3 of 7 — Tools

*It can finally look things up. Watch it change its mind.*

**New here:** Six tools over the college database, and the agent that uses them.

- Now it looks things up. The hall-ticket ticket should route to
- accounts, not examinations — the block is unpaid fees, and only
- the tool call reveals that.
- Read the docstrings in tools.py: they are prompt text, not comments.

**The diff is the lesson.** Everything that changed between step 2 and step 3. The file summary comes first — read that, pick the file that looks interesting, then find it in the diff underneath.

In [ ]:
!git switch -q step-3-tools
!git --no-pager diff --stat step-2-structured step-3-tools
print()
!git --no-pager diff step-2-structured step-3-tools -- src labs evals tests ':!src/college_agent/data' ':!src/college_agent/kb'

In [ ]:
!python labs/lab2_workflow/01_structured_triage.py

---

# Step 4 of 7 — Guardrails

*The difference between “please don't” and “you can't”.*

**New here:** Checks that run before the model — PII, and third-party record requests.

- An instruction is advice; a guardrail is a rule. The agent was
- already told to refuse third-party requests. Now it cannot comply
- even if a cleverly worded ticket talks it into trying.
- They run before the API call, so a blocked ticket costs zero quota.

**The diff is the lesson.** Everything that changed between step 3 and step 4. The file summary comes first — read that, pick the file that looks interesting, then find it in the diff underneath.

In [ ]:
!git switch -q step-4-guardrails
!git --no-pager diff --stat step-3-tools step-4-guardrails
print()
!git --no-pager diff step-3-tools step-4-guardrails -- src labs evals tests ':!src/college_agent/data' ':!src/college_agent/kb'

In [ ]:
!python -m pytest -m 'not live' -q

---

# Step 5 of 7 — One agent, or several

*Four agents instead of one. Roughly double the bill. Was it worth it?*

**New here:** A router in front of three specialists.

- This is a comparison, not an upgrade. The team roughly doubles
- your model calls — router, then specialist.
- Ask honestly whether it did better, or just cost more. For six
- tools and one domain, one agent is usually enough.

**The diff is the lesson.** Everything that changed between step 4 and step 5. The file summary comes first — read that, pick the file that looks interesting, then find it in the diff underneath.

In [ ]:
!git switch -q step-5-team
!git --no-pager diff --stat step-4-guardrails step-5-team
print()
!git --no-pager diff step-4-guardrails step-5-team -- src labs evals tests ':!src/college_agent/data' ':!src/college_agent/kb'

In [ ]:
!python labs/lab2_workflow/01_structured_triage.py
!python labs/lab2_workflow/02_routing_team.py

---

# Step 6 of 7 — Proving it works

*Stop trusting your eyes. Start counting.*

**New here:** Ten golden cases, tool-call reliability, and a deliberate sabotage.

- Reliability first: did it look things up, or get lucky? That check
- is free and deterministic, and catches what accuracy scoring cannot.
- Then break the agent on purpose. If the score does not move, your
- eval suite is decoration.

**The diff is the lesson.** Everything that changed between step 5 and step 6. The file summary comes first — read that, pick the file that looks interesting, then find it in the diff underneath.

In [ ]:
!git switch -q step-6-evals
!git --no-pager diff --stat step-5-team step-6-evals
print()
!git --no-pager diff step-5-team step-6-evals -- src labs evals tests ':!src/college_agent/data' ':!src/college_agent/kb'

In [ ]:
!python labs/lab3_eval/01_reliability.py
!python labs/lab3_eval/02_accuracy.py
!python labs/lab3_eval/03_break_it.py

---

# Step 7 of 7 — Shipping it

*Something someone else can actually call.*

**New here:** A FastAPI service, AgentOS mounted onto it, a Dockerfile and CI.

- /health never calls the model — a health check that costs an API
- request reports your provider being down as you being down.
- The service starts with no API key and reports itself degraded
- rather than crash-looping.
- Three outcomes, three codes: 403 refused, 502 provider down, 200 ok.

**The diff is the lesson.** Everything that changed between step 6 and step 7. The file summary comes first — read that, pick the file that looks interesting, then find it in the diff underneath.

In [ ]:
!git switch -q step-7-deploy
!git --no-pager diff --stat step-6-evals step-7-deploy
print()
!git --no-pager diff step-6-evals step-7-deploy -- src labs evals tests ':!src/college_agent/data' ':!src/college_agent/kb'

In [ ]:
!python labs/lab4_deploy/01_call_the_api.py

---

# Done

You built an agent that reads a complaint, looks up the facts, routes on the
cause instead of the symptom, refuses what it should refuse, and can tell you
how often it gets that right. `git switch main` is the same thing with the
whole history behind it.

Worth noticing what actually did the work. Almost none of it was the model.
It was a schema, six tools with well-written docstrings, two guardrails, and
nine test cases that disagreed with you.

### Two things this notebook could not show you

- **Docker.** Step 7 ships a `Dockerfile` and CI, and neither runs in Colab.
  Read `Dockerfile` and `.github/workflows/ci.yml` — they are short, and the
  comments explain the two-tier design: tests with no API key on every push,
  real agent evals only when a key is configured.
- **The AgentOS web UI.** `make serve` runs a real FastAPI service; in Colab
  the lab drives it with `TestClient` instead, which exercises the same code
  but gives you no browser to click in.

### Where to go next

- Add a case to `evals/cases.py` that the agent gets **wrong**. Much harder
  than one it gets right, and much more useful.
- Delete the "route on the underlying cause" line in
  `src/college_agent/agent.py`, re-run step 3, and watch the hall-ticket
  ticket go to the wrong department. One sentence was holding that up.
- Add a field to `TriageResult` and re-run. The agent fills it in without
  being asked, because the field description *is* the instruction.
- Run the same eval against a stronger model, then decide whether the
  difference was worth the money. That question is the job.

> **Your Colab session is temporary.** Anything you changed here disappears
> when the runtime recycles. If you did something you want to keep, download
> it or push it to your own fork now.